# turkish_distill: TR-MTEB evaluation

Evaluates any embedding model (teachers, baselines, our trained students) on TR-MTEB (`MTEB(Turkish)`, 26 tasks) and compares the finished runs. This notebook is only for evaluation; training lives in `colab_pipeline.ipynb`.

- Results go to Drive: `bitirme/eval/<label>/` (one folder per model, one JSON per task). An interrupted run resumes and skips finished tasks.
- Models are downloaded to the Colab local disk, never to Drive.
- Default protocol: plain encoding, **no instruction/prefix** (this is how the TR-MTEB authors evaluate). Prefixed variants are separate presets with their own labels.
- Use a **separate runtime** from the one doing teacher encoding: both need the GPU.

Before the first run: `HF_TOKEN` in Colab Secrets.

## 1. Setup

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'
EVAL_DIR = f'{DRIVE_ROOT}/eval'

os.makedirs(EVAL_DIR, exist_ok=True)
# HF cache on the LOCAL disk (Drive corrupts large model files).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
# The TR-MTEB authors' fork of MTEB. It pins datasets<3, so pip may downgrade `datasets` in this runtime.
!pip install --quiet "git+https://github.com/selmanbaysan/mteb_tr.git"
!git log -1 --oneline

Check that the Turkish benchmark is available. If this fails, restart the runtime once, then **re-run the Setup cell above again** (a restart clears all variables such as `DRIVE_ROOT` and `EVAL_DIR`) before continuing:

In [ ]:
import mteb
print('mteb', getattr(mteb, '__version__', '?'))
bench = mteb.get_benchmark('MTEB(Turkish)')
print(len(list(bench.tasks)), 'tasks')

## 2. Model presets

Revisions are Hugging Face commit hashes fetched on 2026-10-11, so every run is reproducible. A preset is just the arguments of `run_evaluation`; add your own entries (e.g. a trained student: `model` = path of its `final/` folder on Drive).

Reference numbers for a sanity check (other papers/cards, their own protocols, so only roughly comparable): Qwen3-Embedding-8B 68.66, mE5-large-instruct 67.47, BGE-M3 67.36 (MoganBert paper, TR-MTEB mean over tasks); `modernbert-tr-embed` 68.14 and `turkish-e5-large` 67.17 (model card). A plain run that is far below these probably means a protocol difference (see the `*_instr` presets).

In [ ]:
INSTR = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

# Prompt-sensitive models are scored twice: plain (the TR-MTEB authors' protocol) and with their
# query/document prefixes (`*_instr`). Prompts only change RETRIEVAL scores (verified on mE5: the 15 other
# tasks are bit-identical), so every `*_instr` preset has `reuse_from=<its plain preset>`: it copies the
# plain run's non-retrieval results and only evaluates the 11 retrieval tasks. Run the plain preset first.
# Only the EmbeddingGemma prefixes below were checked against its model card; the Qwen3 instruction was
# written from memory. Models without an `_instr` preset are run plain only, so their retrieval score may
# understate what they reach with the prefixes they expect.
PRESETS = {
    # --- teacher candidates ---
    'bge_m3': dict(model='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181', dtype='float16'),
    'me5_large_instruct': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16'),
    'me5_large_instruct_instr': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16', query_prompt=INSTR,
                                     reuse_from='me5_large_instruct'),
    'qwen3_8b': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16),
    'qwen3_8b_instr': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_8b'),
    'turkish_e5_large': dict(model='ytu-ce-cosmos/turkish-e5-large', revision='02e2362d503bbdeafcb17143b2165c0743f9fdb1', dtype='float16'),

    # --- more teacher candidates (added 2026-10-11) ---
    # Gated: accept the license on the model page with the same Hugging Face account as HF_TOKEN.
    'embeddinggemma_300m': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16'),
    'embeddinggemma_300m_instr': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16',
                                      query_prompt='task: search result | query: ', passage_prompt='title: none | text: ',
                                      reuse_from='embeddinggemma_300m'),
    'harrier_0_6b': dict(model='microsoft/harrier-oss-v1-0.6b', revision='f9b9dc8d367d443f2479d27aa5d8d2850c0774ee', dtype='bfloat16'),
    'harrier_270m': dict(model='microsoft/harrier-oss-v1-270m', revision='31de22b673913c7d658c0f03f792d77c2dcf8ebd', dtype='bfloat16'),
    'gte_multilingual_base': dict(model='Alibaba-NLP/gte-multilingual-base', revision='9bbca17d9273fd0d03d5725c7a4b0f6b45142062', dtype='float16', trust_remote_code=True),
    'arctic_embed_l_v2': dict(model='Snowflake/snowflake-arctic-embed-l-v2.0', revision='ac6544c8a46e00af67e330e85a9028c66b8cfd9a', dtype='float16'),
    'nomic_embed_v2_moe': dict(model='nomic-ai/nomic-embed-text-v2-moe', revision='1066b6599d099fbb93dfcb64f9c37a7c9e503e85', dtype='float32', trust_remote_code=True),
    'qwen3_4b': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16),
    'qwen3_4b_instr': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_4b'),
    'qwen3_0_6b': dict(model='Qwen/Qwen3-Embedding-0.6B', revision='97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3', dtype='bfloat16'),

    # --- baselines from the same lab ---
    'modernbert_tr_embed': dict(model='ytu-ce-cosmos/modernbert-tr-embed', revision='855820e0079102805908d60d32cd9fd59c970b80'),
    'modernbert_tr_1k_embed': dict(model='ytu-ce-cosmos/modernbert-tr-1k-embed', revision='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    # --- our student before any distillation (raw MLM encoder + mean pooling) ---
    'modernbert_tr_base_1k_raw': dict(model='ytu-ce-cosmos/modernbert-tr-base-1k', revision='c2e35f553101cce5b03fbb318ef8728473fd34dd'),

    # --- our trained students (stage 1): `model` is the run's final/ folder on Drive. Same protocol as the raw
    # student above (plain encoding, mean pooling), so the numbers are directly comparable. Add one line per run.
    'c_mnrl_s20_b128': dict(model=f'{DRIVE_ROOT}/runs/c_modernbert-tr-base-1k_mnrl_s20_b128_chunk32/20261018-053836-3551f77/final'),
}
print(list(PRESETS))

In [ ]:
# Stage-1 distilled students: the final/ folder of each run on Drive (same plain protocol as every other preset).
# d_single_*: one teacher; d_two_*: both teachers, equal weights; d_two_bge*_qwen*: static weight ratios BGE:Qwen;
# d_two_qwenramp900: Qwen's weight rises from 0 to 1 over the first 900 of 1800 steps;
# d_single_me5 / d_two_bge_m3_me5: multilingual-e5-large-instruct alone / with BGE-M3 (equal weights).
# Evaluate only finished runs (final/ exists).
PRESETS.update({
    'd_single_qwen3_8b': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_qwen3_8b/20261018-094046-e346476/final'),
    'd_single_bge_m3': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_bge_m3/20261018-094050-e346476/final'),
    'd_two_qwen3_8b_bge_m3': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_qwen3_8b_bge_m3/20261018-094149-e346476/final'),
    'd_two_bge3_qwen1': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge3_qwen1/20261018-134441-4fde652/final'),
    'd_two_bge1_qwen3': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge1_qwen3/20261018-134500-4fde652/final'),
    'd_two_qwenramp900': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_qwenramp900/20261018-134505-4fde652/final'),
    'd_single_me5': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_me5/20261019-051608-852baf8/final'),
    'd_two_bge_m3_me5': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge_m3_me5/20261019-051622-852baf8/final'),
})
# The same students with a Normalize layer appended (suffix _norm). A folder saved by training is loaded as transformer
# + mean pooling only, so its vectors are not unit length. That matters for the classification and clustering tasks
# (a logistic regression / k-means is fitted on the raw vectors; sklearn's "lbfgs failed to converge" warnings in the
# log are a symptom), while retrieval, STS and pair classification use cosine and do not care. BGE-M3, mE5, Qwen3 and
# modernbert-tr-embed ship a Normalize layer; modernbert-tr-1k-embed does not.
PRESETS.update({f'{k}_norm': {**v, 'normalize': True} for k, v in list(PRESETS.items()) if k.startswith(('c_', 'd_'))})
print([k for k in PRESETS if k.startswith(('c_', 'd_')) and not k.endswith('_norm')])

## 3. Run one evaluation

`TASKS = None` runs all 26 tasks. For a quick smoke test use a few small tasks; they are stored under a separate label (`<name>__subset`) so they never mix with full runs.

In [ ]:
CHOICE = 'c_mnrl_s20_b128'
TASKS = None   # e.g. ['STSbTR', 'TurkishIronyClassification', 'TurkishMovieSentimentClassification']

# Guard: this cell needs the Setup cell (section 1) to have run in THIS runtime session
# (it defines EVAL_DIR and PRESETS comes from section 2). If the package is missing but the
# repo is there, install it now instead of failing with ModuleNotFoundError.


from turkish_distill.eval.trmteb import run_evaluation, summarize_dir, format_table, format_group_table

label = CHOICE if TASKS is None else f'{CHOICE}__subset'
label_dir = run_evaluation(output_dir=EVAL_DIR, label=label, task_names=TASKS, **PRESETS[CHOICE])
result = {label: summarize_dir(label_dir)}
print(format_table(result))
print()
print(format_group_table(result))

In [ ]:
import importlib.util, os, subprocess, sys
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
if importlib.util.find_spec('turkish_distill') is None:
    if not os.path.isdir(f'{REPO_DIR}/src'):
        raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')

In [ ]:
# Run several presets one after another (e.g. before going away). Each model runs in its OWN Python process:
# if one fails (gated access, incompatible remote code, a CUDA "device-side assert"...) the error is reported
# and the next model starts in a clean process. An interrupted preset resumes: finished tasks are skipped.
# Put the plain preset before its `*_instr` variant (the variant reuses the plain run's 15 non-retrieval results).
#
# Already done (2026-10-11): bge_m3, me5_large_instruct(+_instr), qwen3_8b, turkish_e5_large, embeddinggemma_300m(+_instr),
# harrier_0_6b, harrier_270m, modernbert_tr_embed, modernbert_tr_1k_embed, modernbert_tr_base_1k_raw.
# Also done: qwen3_4b(+_instr), qwen3_0_6b, c_mnrl_s20_b128 (contrastive baseline), d_single_qwen3_8b, d_single_bge_m3,
# d_two_qwen3_8b_bge_m3, d_two_bge3_qwen1, d_two_bge1_qwen3, d_two_qwenramp900 (stage-1 distilled students).
# Known incompatible with the current transformers: gte_multilingual_base (its remote code calls a removed method).
BATCH = ['d_single_me5', 'd_two_bge_m3_me5']   # the mE5 students (finished runs already evaluated are skipped)

import time
from turkish_distill.eval.trmteb import run_evaluation_isolated, summarize_dir, format_table

failed = {}
for name in BATCH:
    t0 = time.time()
    print(f'\n########## {name}')
    try:
        d = run_evaluation_isolated(label=name, output_dir=EVAL_DIR, **PRESETS[name])
        print(f'\n=== {name} done in {(time.time() - t0) / 60:.1f} min')
        print(format_table({name: summarize_dir(d)}))
    except Exception as exc:
        failed[name] = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'\n=== {name} FAILED after {(time.time() - t0) / 60:.1f} min: {failed[name]}')

print('\nfailed:', failed or 'none')

In [ ]:
# Does normalising the vectors change the scale-sensitive scores? Re-score ONLY the 10 classification + clustering
# tasks of a few students with a Normalize layer and compare with their full (un-normalised) evaluation. The other
# 16 tasks use cosine similarity and cannot change, so they are not re-run. Results go to <name>__subset folders
# (never mixed with full runs).
CHECK = ['d_single_bge_m3', 'c_mnrl_s20_b128', 'd_single_me5']

from turkish_distill.eval.trmteb import SCALE_SENSITIVE_TASKS, TASK_CATEGORIES, run_evaluation_isolated, summarize_dir

for name in CHECK:
    run_evaluation_isolated(label=f'{name}_norm__subset', output_dir=EVAL_DIR, task_names=SCALE_SENSITIVE_TASKS, **PRESETS[f'{name}_norm'])

for name in CHECK:
    before = summarize_dir(f'{EVAL_DIR}/{name}')['per_task']
    after = summarize_dir(f'{EVAL_DIR}/{name}_norm__subset')['per_task']
    print(f'\n{name}: un-normalised -> normalised')
    for category in ('Classification', 'Clustering'):
        tasks = [t for t in SCALE_SENSITIVE_TASKS if TASK_CATEGORIES[t] == category]
        b, a = sum(before[t] for t in tasks) / len(tasks), sum(after[t] for t in tasks) / len(tasks)
        print(f'  {category:15s} {b:6.2f} -> {a:6.2f}  ({a - b:+.2f})')
    worst = max(SCALE_SENSITIVE_TASKS, key=lambda t: abs(after[t] - before[t]))
    print(f'  largest single-task change: {worst} {before[worst]:.1f} -> {after[worst]:.1f}')

## 4. Compare all finished evaluations

In [ ]:
import glob
import pandas as pd
from turkish_distill.eval.trmteb import format_group_table, format_table, summarize_dir

dirs = sorted(d for d in glob.glob(f'{EVAL_DIR}/*') if os.path.isdir(d) and not d.endswith('__subset'))
summaries = {os.path.basename(d): summarize_dir(d) for d in dirs}
print(format_table(summaries))
print()
print(format_group_table(summaries))   # the nine skill groups (see TASK_GROUPS in eval/trmteb.py)

per_task = pd.DataFrame({k: v['per_task'] for k, v in summaries.items()}).round(2)
per_task